# V_CRG_STUDENT_COURSE — Cleaning Notebook

This notebook cleans the student course attempts table from `V_CRG_STUDENT_COURSE`.

The output table is `clean_student_course_attempts`.

The table grain is one row per `student_course_id`.

`clean_student_course_attempts` represents completed course attempts with real pass/fail academic outcomes.

Withdrawn courses are excluded from this clean table and stored in `audit_removed_withdrawn`.

Withdrawn history can be modeled later in a separate feature table if needed.

It is not a course-offer table and not an eligibility table.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.cleaning_utils import normalize_id_columns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

In [ ]:
VERSION = "v1"

# Fill DATA_PATH manually with the source .parquet or .csv file path before running the loading cell.
DATA_PATH = r"D:\AI\Real projects\Academic_Advisor\data\raw\v_crg_student_course_raw.parquet"

In [ ]:
if not DATA_PATH:
    raise ValueError("DATA_PATH is empty. Fill DATA_PATH manually with the source .parquet or .csv file path before running this cell.")

data_path = Path(DATA_PATH)
suffix = data_path.suffix.lower()

if suffix == ".parquet":
    df_raw = pd.read_parquet(data_path)
elif suffix == ".csv":
    df_raw = pd.read_csv(data_path, dtype="string")
else:
    raise ValueError("DATA_PATH must point to a .parquet or .csv file.")

## Initial Inspection

In [ ]:
df_raw.shape

In [ ]:
df_raw.columns.tolist()

In [ ]:
df_raw.info()

In [ ]:
df_raw.head(5)

In [ ]:
null_report = (
    df_raw.isna()
    .sum()
    .rename("null_count")
    .to_frame()
    .assign(null_rate=lambda x: x["null_count"] / len(df_raw))
    .sort_values("null_count", ascending=False)
)

null_report

In [ ]:
nunique_report = (
    df_raw.nunique(dropna=False)
    .rename("nunique_including_null")
    .to_frame()
    .sort_values("nunique_including_null", ascending=False)
)

nunique_report

In [ ]:
df_raw["register_status"].value_counts(dropna=False)

In [ ]:
df_raw["finish_status"].value_counts(dropna=False)

In [ ]:
df_raw["active"].value_counts(dropna=False)

## Helper Functions

In [ ]:
def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with lowercase snake_case column names."""
    normalized_columns = (
        pd.Index(df.columns)
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True)
        .str.strip("_")
        .str.lower()
    )
    return df.copy().set_axis(normalized_columns, axis=1)


def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None: 
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")


def normalize_text_series(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    return cleaned.mask(cleaned == "", pd.NA)


def normalize_code_series(series: pd.Series) -> pd.Series:
    return normalize_text_series(series).str.upper()


def convert_nullable_integer(series: pd.Series, column_name: str) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    source_has_value = series.notna() & (series.astype("string").str.strip() != "")
    invalid_values = source_has_value & numeric.isna()

    if invalid_values.any():
        examples = series.loc[invalid_values].drop_duplicates().head(10).tolist()
        raise ValueError(f"{column_name} contains non-numeric values that cannot be converted to Int64. Examples: {examples}")

    numeric_values = numeric.dropna()
    non_integer_values = numeric_values[~np.isclose(numeric_values % 1, 0, atol=1e-9)]

    if not non_integer_values.empty:
        examples = series.loc[non_integer_values.index].drop_duplicates().head(10).tolist()
        raise ValueError(f"{column_name} contains non-integer values that cannot be converted to Int64. Examples: {examples}")

    return numeric.round().astype("Int64")


def validate_final_mark_is_integer(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    source_has_value = series.notna() & (series.astype("string").str.strip() != "")
    invalid_values = source_has_value & numeric.isna()

    if invalid_values.any():
        examples = series.loc[invalid_values].drop_duplicates().head(10).tolist()
        raise ValueError(f"final_mark contains non-numeric values. Examples: {examples}")

    numeric_values = numeric.dropna()
    non_integer_values = numeric_values[~np.isclose(numeric_values % 1, 0, atol=1e-9)]

    if not non_integer_values.empty:
        examples = series.loc[non_integer_values.index].drop_duplicates().head(10).tolist()
        raise ValueError(f"final_mark contains non-integer decimals. Examples: {examples}")

    return numeric.round().astype("Int64")


def create_audit_summary(audit_frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"audit_table": audit_name, "rows": len(audit_df), "columns": audit_df.shape[1]}
            for audit_name, audit_df in audit_frames.items()
        ]
    )

## Cleaning Function

In [ ]:
EXPECTED_SOURCE_COLUMNS = [
    "student_course_id",
    "student_id",
    "course_id",
    "part_id",
    "grade_id",
    "final_mark",
    "points",
    "finish_status",
    "course_name_sl",
    "register_status",
    "in_credits",
    "in_gpa",
    "in_agpa",
    "study_mode",
    "degree_id",
    "student_name_sl",
    "degree_name_sl",
    "faculty_id",
    "course_credits",
    "active",
]

EXPECTED_CLEAN_COLUMNS = [
    "student_course_id",
    "student_id",
    "course_id",
    "part_id",
    "degree_id",
    "faculty_id",
    "grade_id",
    "final_mark",
    "points",
    "finish_status",
    "course_outcome_status",
    "register_status",
    "in_credits",
    "in_gpa",
    "in_agpa",
    "study_mode",
    "course_credits",
    "course_name_sl",
    "degree_name_sl",
    "attempt_number",
    "attempt_count",
]

VALID_REGISTER_STATUSES = {"R", "E"}
VALID_FINISH_STATUSES = {"P", "F", "FE", "FA"}
VALID_COURSE_OUTCOME_STATUSES = {"passed", "failed"}


def clean_v_crg_student_course(df_raw: pd.DataFrame) -> dict:
    df = normalize_column_names(df_raw)
    validate_required_columns(df, EXPECTED_SOURCE_COLUMNS)

    raw_rows = len(df)

    df["active"] = normalize_code_series(df["active"])
    active_mask = df["active"].eq("A")
    audit_removed_inactive = df.loc[~active_mask].copy()
    df_active = df.loc[active_mask].copy()
    rows_after_active_filter = len(df_active)

    df_active["register_status"] = normalize_code_series(df_active["register_status"])
    register_status_mask = df_active["register_status"].isin(VALID_REGISTER_STATUSES)
    audit_removed_register_status = df_active.loc[~register_status_mask].copy()
    df_registered = df_active.loc[register_status_mask].copy()
    rows_after_register_status_filter = len(df_registered)

    df_registered["finish_status"] = normalize_code_series(df_registered["finish_status"])
    withdrawn_mask = df_registered["finish_status"].eq("W").fillna(False)
    audit_removed_withdrawn = df_registered.loc[withdrawn_mask].copy()
    df_non_withdrawn = df_registered.loc[~withdrawn_mask].copy()

    finish_status_mask = df_non_withdrawn["finish_status"].isin(VALID_FINISH_STATUSES)
    audit_removed_finish_status = df_non_withdrawn.loc[~finish_status_mask].copy()
    df_finished = df_non_withdrawn.loc[finish_status_mask].copy()
    rows_after_finish_status_filter = len(df_finished)

    cleaned = df_finished.copy()

    id_columns = ["student_course_id", "student_id", "course_id", "degree_id", "faculty_id", "grade_id"]
    cleaned = normalize_id_columns(cleaned, id_columns)

    cleaned["part_id"] = convert_nullable_integer(cleaned["part_id"], "part_id")
    cleaned["final_mark"] = validate_final_mark_is_integer(cleaned["final_mark"])
    cleaned["points"] = pd.to_numeric(cleaned["points"], errors="coerce").astype("Float64")
    cleaned["course_credits"] = pd.to_numeric(cleaned["course_credits"], errors="coerce").astype("Float64")

    for column in ["course_name_sl", "degree_name_sl"]:
        cleaned[column] = normalize_text_series(cleaned[column])

    for column in ["in_credits", "in_gpa", "in_agpa", "study_mode"]:
        cleaned[column] = normalize_code_series(cleaned[column])

    critical_columns = ["student_course_id", "student_id", "course_id", "part_id", "course_credits"]
    missing_critical_mask = cleaned[critical_columns].isna().any(axis=1)
    audit_removed_missing_critical = cleaned.loc[missing_critical_mask].copy()
    cleaned = cleaned.loc[~missing_critical_mask].copy()

    bad_course_credits_mask = cleaned["course_credits"].le(0)
    audit_removed_bad_course_credits = cleaned.loc[bad_course_credits_mask].copy()
    cleaned = cleaned.loc[~bad_course_credits_mask].copy()

    bad_final_mark_mask = cleaned["final_mark"].notna() & ~cleaned["final_mark"].between(0, 100)
    audit_removed_bad_final_mark = cleaned.loc[bad_final_mark_mask].copy()
    cleaned = cleaned.loc[~bad_final_mark_mask].copy()

    outcome_map = {
        "P": "passed",
        "F": "failed",
        "FE": "failed",
        "FA": "failed",
    }
    cleaned["course_outcome_status"] = cleaned["finish_status"].map(outcome_map).astype("string")

    cleaned = cleaned.sort_values(
        ["student_id", "course_id", "part_id", "student_course_id"],
        kind="mergesort",
    ).reset_index(drop=True)

    attempt_group = cleaned.groupby(["student_id", "course_id"], dropna=False, sort=False)
    cleaned["attempt_number"] = (attempt_group.cumcount() + 1).astype("Int64")
    cleaned["attempt_count"] = attempt_group["student_course_id"].transform("size").astype("Int64")

    for column in ["finish_status", "course_outcome_status", "register_status", "in_credits", "in_gpa", "in_agpa", "study_mode"]:
        cleaned[column] = cleaned[column].astype("category")

    clean_student_course_attempts = cleaned[EXPECTED_CLEAN_COLUMNS].copy()

    duplicate_student_course_id_count = int(clean_student_course_attempts["student_course_id"].duplicated().sum())
    duplicate_student_course_part_count = int(
        clean_student_course_attempts.duplicated(["student_id", "course_id", "part_id"]).sum()
    )
    null_final_mark_rows = int(clean_student_course_attempts["final_mark"].isna().sum())

    cleaning_summary = pd.DataFrame(
        [
            {
                "raw_rows": raw_rows,
                "rows_after_active_filter": rows_after_active_filter,
                "rows_after_register_status_filter": rows_after_register_status_filter,
                "rows_after_finish_status_filter": rows_after_finish_status_filter,
                "final_clean_rows": len(clean_student_course_attempts),
                "final_clean_rows_after_dropping_withdrawn": len(clean_student_course_attempts),
                "removed_inactive_rows": len(audit_removed_inactive),
                "removed_register_status_rows": len(audit_removed_register_status),
                "removed_withdrawn_rows": len(audit_removed_withdrawn),
                "removed_finish_status_rows": len(audit_removed_finish_status),
                "removed_missing_critical_rows": len(audit_removed_missing_critical),
                "removed_bad_course_credits_rows": len(audit_removed_bad_course_credits),
                "removed_bad_final_mark_rows": len(audit_removed_bad_final_mark),
                "unique_students": clean_student_course_attempts["student_id"].nunique(dropna=True),
                "unique_courses": clean_student_course_attempts["course_id"].nunique(dropna=True),
                "unique_semesters": clean_student_course_attempts["part_id"].nunique(dropna=True),
                "unique_degrees": clean_student_course_attempts["degree_id"].nunique(dropna=True),
                "duplicate_student_course_id_count": duplicate_student_course_id_count,
                "duplicate_student_course_part_count": duplicate_student_course_part_count,
                "null_final_mark_rows": null_final_mark_rows,
            }
        ]
    )

    if duplicate_student_course_id_count > 0:
        raise ValueError(f"student_course_id must be unique. Duplicate row count: {duplicate_student_course_id_count}")

    null_counts = clean_student_course_attempts[critical_columns].isna().sum()
    if null_counts.any():
        raise ValueError(f"Clean output contains nulls in critical columns: {null_counts[null_counts > 0].to_dict()}")

    actual_register_statuses = set(clean_student_course_attempts["register_status"].dropna().astype("string"))
    if not actual_register_statuses.issubset(VALID_REGISTER_STATUSES):
        raise ValueError(f"Unexpected register_status values: {sorted(actual_register_statuses - VALID_REGISTER_STATUSES)}")

    actual_finish_statuses = set(clean_student_course_attempts["finish_status"].dropna().astype("string"))
    if not actual_finish_statuses.issubset(VALID_FINISH_STATUSES):
        raise ValueError(f"Unexpected finish_status values: {sorted(actual_finish_statuses - VALID_FINISH_STATUSES)}")

    actual_outcome_statuses = set(clean_student_course_attempts["course_outcome_status"].dropna().astype("string"))
    if not actual_outcome_statuses.issubset(VALID_COURSE_OUTCOME_STATUSES):
        raise ValueError(f"Unexpected course_outcome_status values: {sorted(actual_outcome_statuses - VALID_COURSE_OUTCOME_STATUSES)}")

    validate_final_mark_is_integer(clean_student_course_attempts["final_mark"])

    out_of_range_final_mark_count = int(
        (clean_student_course_attempts["final_mark"].notna() & ~clean_student_course_attempts["final_mark"].between(0, 100)).sum()
    )
    if out_of_range_final_mark_count:
        raise ValueError(f"final_mark must be between 0 and 100 when not null. Bad row count: {out_of_range_final_mark_count}")

    if null_final_mark_rows:
        raise ValueError(
            "final_mark must not be null in clean_student_course_attempts after withdrawn rows are excluded. "
            f"Bad row count: {null_final_mark_rows}."
        )

    non_positive_course_credits_count = int(clean_student_course_attempts["course_credits"].le(0).sum())
    if non_positive_course_credits_count:
        raise ValueError(f"course_credits must be greater than 0. Bad row count: {non_positive_course_credits_count}")

    return {
        "clean_student_course_attempts": clean_student_course_attempts,
        "audit_removed_inactive": audit_removed_inactive,
        "audit_removed_register_status": audit_removed_register_status,
        "audit_removed_withdrawn": audit_removed_withdrawn,
        "audit_removed_finish_status": audit_removed_finish_status,
        "audit_removed_missing_critical": audit_removed_missing_critical,
        "audit_removed_bad_course_credits": audit_removed_bad_course_credits,
        "audit_removed_bad_final_mark": audit_removed_bad_final_mark,
        "cleaning_summary": cleaning_summary,
    }

## Run Cleaning

In [ ]:
results = clean_v_crg_student_course(df_raw)

clean_student_course_attempts = results["clean_student_course_attempts"]
audit_removed_inactive = results["audit_removed_inactive"]
audit_removed_register_status = results["audit_removed_register_status"]
audit_removed_withdrawn = results["audit_removed_withdrawn"]
audit_removed_finish_status = results["audit_removed_finish_status"]
audit_removed_missing_critical = results["audit_removed_missing_critical"]
audit_removed_bad_course_credits = results["audit_removed_bad_course_credits"]
audit_removed_bad_final_mark = results["audit_removed_bad_final_mark"]
cleaning_summary = results["cleaning_summary"]

## Output Inspection

In [ ]:
cleaning_summary

In [ ]:
clean_student_course_attempts.shape

In [ ]:
clean_student_course_attempts.head(20)

In [ ]:
clean_student_course_attempts['in_gpa'].value_counts(dropna=False)

In [ ]:
clean_student_course_attempts.info()

In [ ]:
clean_null_report = (
    clean_student_course_attempts.isna()
    .sum()
    .rename("null_count")
    .to_frame()
    .assign(null_rate=lambda x: x["null_count"] / len(clean_student_course_attempts))
    .sort_values("null_count", ascending=False)
)

clean_null_report

In [ ]:
clean_student_course_attempts["register_status"].value_counts(dropna=False)

In [ ]:
clean_student_course_attempts["finish_status"].value_counts(dropna=False)

In [ ]:
clean_student_course_attempts["course_outcome_status"].value_counts(dropna=False)

In [ ]:
clean_student_course_attempts["in_gpa"].value_counts(dropna=False)

In [ ]:
clean_student_course_attempts["in_agpa"].value_counts(dropna=False)

In [ ]:
clean_student_course_attempts["in_credits"].value_counts(dropna=False)

In [ ]:
clean_student_course_attempts.duplicated(["student_id", "course_id", "part_id"]).sum()

## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(
    data=clean_student_course_attempts,
    x="final_mark",
    hue="course_outcome_status",
    bins=20,
    multiple="stack",
    ax=ax,
)
ax.set_title("Final Mark Distribution")
ax.set_xlabel("Final mark")
ax.set_ylabel("Attempt count")
plt.tight_layout()
plt.show()

In [ ]:
students_by_part = (
    clean_student_course_attempts.groupby("part_id", observed=True)["student_id"]
    .nunique()
    .reset_index(name="student_count")
    .sort_values("part_id")
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=students_by_part, x="part_id", y="student_count", marker="o", ax=ax)
ax.set_title("Students With Completed Attempts by Semester")
ax.set_xlabel("Part ID")
ax.set_ylabel("Unique students")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
courses_per_student_semester = (
    clean_student_course_attempts.groupby(["student_id", "part_id"], observed=True)["course_id"]
    .nunique()
    .reset_index(name="course_count")
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=courses_per_student_semester, x="course_count", color="#4C78A8", ax=ax)
ax.set_title("Courses per Student in the Same Semester")
ax.set_xlabel("Unique courses in semester")
ax.set_ylabel("Student-semester count")
plt.tight_layout()
plt.show()

## Audit Inspection

In [ ]:
audit_frames = {
    "audit_removed_inactive": audit_removed_inactive,
    "audit_removed_register_status": audit_removed_register_status,
    "audit_removed_withdrawn": audit_removed_withdrawn,
    "audit_removed_finish_status": audit_removed_finish_status,
    "audit_removed_missing_critical": audit_removed_missing_critical,
    "audit_removed_bad_course_credits": audit_removed_bad_course_credits,
    "audit_removed_bad_final_mark": audit_removed_bad_final_mark,
}

create_audit_summary(audit_frames)

In [ ]:
audit_removed_inactive.shape

In [ ]:
audit_removed_inactive.head(20)

In [ ]:
audit_removed_register_status.shape

In [ ]:
audit_removed_register_status.head(20)

In [ ]:
audit_removed_withdrawn.shape

In [ ]:
audit_removed_withdrawn.head(20)

In [ ]:
audit_removed_finish_status.shape

In [ ]:
audit_removed_finish_status.head(20)

In [ ]:
audit_removed_missing_critical.shape

In [ ]:
audit_removed_missing_critical.head(20)

In [ ]:
audit_removed_bad_course_credits.shape

In [ ]:
audit_removed_bad_course_credits.head(20)

In [ ]:
audit_removed_bad_final_mark.shape

In [ ]:
audit_removed_bad_final_mark.head(20)

This notebook creates the clean completed-attempts core table only.

`clean_student_course_attempts` represents completed course attempts with real pass/fail academic outcomes.

Withdrawn courses are excluded from this clean table and stored in `audit_removed_withdrawn`.

Withdrawn history can be modeled later in a separate feature table if needed.

It does not create:

- `passed_before`
- `failed_before`
- candidate courses
- course difficulty
- model features
- recommendations

In [ ]:
clean_student_course_attempts.head()

In [ ]:
clean_student_course_attempts.shape

In [ ]:
students_by_part = (
    clean_student_course_attempts.groupby("faculty_id", observed=True)["student_id"]
    .nunique()
    .reset_index(name="student_count")
    .sort_values("faculty_id")
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=students_by_part, x="faculty_id", y="student_count", marker="o", ax=ax)
ax.set_title("Students With Completed Attempts by Semester")
ax.set_xlabel("Faculty ID")
ax.set_ylabel("Unique students")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
clean_student_course_attempts['faculty_id'].value_counts()

In [ ]:
clean_student_course_attempts['course_outcome_status'].value_counts()

In [ ]:
# clean_student_course_attempts.to_parquet(r"D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_CRG_STUDENT_COURSE\clean_v_crg_student_course.parquet", index=False)